## Monte Carlo Policy Control

In [ ]:
import gymnasium as gym
import numpy as np
from collections import defaultdict
from tqdm.notebook import trange
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create Blackjack environment
env = gym.make('Blackjack-v1')

In [ ]:
def random_policy(observation):
    """A random policy: stick or hit with equal probability"""
    return np.random.choice([0, 1])

def generate_episode_exploring_starts(env, policy):
    """Generate an episode with exploring starts"""
    episode = []
    observation, _ = env.reset()
    done = False
    
    # Randomly select action
    action = np.random.choice([0, 1])
    
    while not done:
        next_observation, reward, terminated, truncated, _ = env.step(action)
        episode.append((observation, action, reward))
        observation = next_observation
        if observation in policy:
            action = policy[observation]
        else:
            action = random_policy(observation)
        done = terminated or truncated
    return episode


In [ ]:
def mc_control_es(env, num_episodes=1000000, gamma=1.0):
    Q = defaultdict(lambda: defaultdict(float))
    returns = defaultdict(lambda: defaultdict(list))
    policy = defaultdict(lambda: np.random.choice([0, 1]))  # Random initial policy

    for _ in trange(num_episodes):
        # Generate an episode using exploring starts
        episode = generate_episode_exploring_starts(env, policy)
        
        # Update Q-values and policy
        state_action_pairs = set()
        for t, (state, action, reward) in enumerate(episode):
            if (state, action) not in state_action_pairs:
                state_action_pairs.add((state, action))
                G = sum([gamma**i * r for i, (_, _, r) in enumerate(episode[t:])])
                returns[state][action].append(G)
                Q[state][action] = np.mean(returns[state][action])
                
                # Update policy (greedy with respect to Q)
                policy[state] = max(Q[state], key=Q[state].get)

    return policy, Q

In [ ]:
# Run Exploring Starts Monte Carlo control
optimal_policy, optimal_q_value_function = mc_control_es(env)

In [ ]:
def plot_q_values(Q, title, filename, cmap='YlGnBu'):
    # Prepare data for plotting
    player_sum = range(12, 22)
    dealer_show = range(1, 11)
    
    stick_values = np.zeros((len(player_sum), len(dealer_show), 2))
    hit_values = np.zeros((len(player_sum), len(dealer_show), 2))
    
    for state in Q:
        player_total, dealer_card, usable_ace = state
        if 12 <= player_total <= 21 and 1 <= dealer_card <= 10:
            stick_values[player_total-12, dealer_card-1, usable_ace] = Q[state][0]
            hit_values[player_total-12, dealer_card-1, usable_ace] = Q[state][1]
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    fig.suptitle(title, fontsize=16)
    
    # Plot heatmaps
    for i, ace in enumerate(['No Usable Ace', 'Usable Ace']):
        sns.heatmap(stick_values[:, :, i], ax=axes[i, 0], cmap=cmap, 
                    xticklabels=dealer_show, yticklabels=player_sum)
        axes[i, 0].set_title(f'Stick - {ace}')
        axes[i, 0].set_xlabel('Dealer Showing')
        axes[i, 0].set_ylabel('Player Sum')
        
        sns.heatmap(hit_values[:, :, i], ax=axes[i, 1], cmap=cmap, 
                    xticklabels=dealer_show, yticklabels=player_sum)
        axes[i, 1].set_title(f'Hit - {ace}')
        axes[i, 1].set_xlabel('Dealer Showing')
        axes[i, 1].set_ylabel('Player Sum')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_q_values(optimal_q_value_function, "Monte Carlo Exploring Starts Q-Values", 'bjk_mces_q_values.png')
plot_q_values(optimal_q_value_function, "Monte Carlo Exploring Starts Q-Values", 'bjk_mces_q_values_bw.png', 'Greys')

In [ ]:
def plot_policy(policy, filename, cmap='cool'):
    # Create arrays to hold the action values
    usable_ace = np.zeros((10, 10), dtype=np.int32)
    no_usable_ace = np.zeros((10, 10), dtype=np.int32)

    # Iterate over all possible states in the Blackjack environment
    for (player_sum, dealer_showing, ace), action in policy.items():
        if 12 <= player_sum <= 21 and 1 <= dealer_showing <= 10:
            player_sum_idx = player_sum - 12
            dealer_showing_idx = dealer_showing - 1
            if ace:
                usable_ace[player_sum_idx, dealer_showing_idx] = action
            else:
                no_usable_ace[player_sum_idx, dealer_showing_idx] = action

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    fig.suptitle('Optimal Policy for Blackjack')

    axes[0].imshow(usable_ace, cmap=cmap, interpolation='none', aspect='auto')
    axes[0].set_title('Usable Ace')
    axes[0].set_xlabel('Dealer Showing')
    axes[0].set_ylabel('Player Sum')
    axes[0].set_xticks(np.arange(10))
    axes[0].set_yticks(np.arange(10))
    axes[0].set_xticklabels([str(i) for i in range(1, 11)])
    axes[0].set_yticklabels([str(i) for i in range(12, 22)])
    for i in range(10):
        for j in range(10):
            text = 'H' if usable_ace[i, j] == 1 else 'S'
            axes[0].text(j, i, text, ha='center', va='center', color='black')

    axes[1].imshow(no_usable_ace, cmap=cmap, interpolation='none', aspect='auto')
    axes[1].set_title('No Usable Ace')
    axes[1].set_xlabel('Dealer Showing')
    axes[1].set_ylabel('Player Sum')
    axes[1].set_xticks(np.arange(10))
    axes[1].set_yticks(np.arange(10))
    axes[1].set_xticklabels([str(i) for i in range(1, 11)])
    axes[1].set_yticklabels([str(i) for i in range(12, 22)])
    for i in range(10):
        for j in range(10):
            text = 'H' if no_usable_ace[i, j] == 1 else 'S'
            axes[1].text(j, i, text, ha='center', va='center', color='black')
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_policy(optimal_policy, 'bjk_es_op.png')

In [ ]:
colors = [(0.5, 0.5, 0.5), (1, 1, 1)]  # Grey to white
n_bins = 100  # Discretizes the interpolation into 100 steps
cmap_name = 'grey_to_white'
cm = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)

In [ ]:
plot_policy(optimal_policy, 'bjk_es_op_bw.png', cm)